In [ ]:
import torch
import torch.nn.functional as F
import os
from vllm import LLM, SamplingParams

# ---------- vLLM config ----------
os.environ.setdefault("VLLM_USE_V1", "1")
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

llm = LLM(model="meta-llama/Meta-Llama-3.1-8B-Instruct", enforce_eager=True, tensor_parallel_size=1)
engine = llm.llm_engine


In [ ]:

# ---------- run a short decode so KV is populated ----------
req_id = "kv_analysis9"
prompt = "Explain how transformer attention mechanisms work with key-value caches."
engine.add_request(req_id, prompt, SamplingParams(max_tokens=25))
print("Processing request...")

for i in range(20):  # a few steps is enough
    result = engine.step()
    if not result:
        print(f"Step {i+1}: No outputs")
        break
    finished = False
    for out in result:
        if out.request_id == req_id and out.finished:
            finished = True
    print(f"Step {i+1}: Generated {len(result)} outputs; finished={finished}")
    if finished:
        break

print("\nRequest processing completed.")

# ---------- scheduler state ----------
core_client = engine.engine_core
engine_core = core_client.engine_core if hasattr(core_client, "engine_core") else None
if engine_core is None:
    raise RuntimeError("This notebook expects an in-process EngineCore.")
scheduler = engine_core.scheduler
print(f"\nScheduler state:\n  Running: {len(scheduler.running)}\n  Waiting: {len(scheduler.waiting)}")

# ---------- discover attention layers from static_forward_context ----------
ctx = engine.vllm_config.compilation_config.static_forward_context
print("\nAvailable attention layers:")
attn_layers = {}
for key in ctx.keys():
    if "self_attn.attn" in key:
        # Example: 'model.layers.24.self_attn.attn' -> index 2 holds layer id
        parts = key.split(".")
        try:
            layer_num = int(parts[2])  # 0-indexed in your earlier parsing
        except Exception:
            continue
        attn_layers[layer_num] = ctx[key]
        print(f"  Layer {layer_num+1}: {key}")
layer_ids = sorted(attn_layers.keys())
if not layer_ids:
    raise RuntimeError("No attention layers discovered.")

# ---------- helpers for KV extraction & similarity ----------
def get_kv_seq(attn_layer, ve=0):
    """
    Return keys_seq, values_seq as [T, H_kv, Dh] tensors (CPU).
    vLLM v1 KV layout: [num_blocks, block_size, H_kv, Dh].
    """
    if not hasattr(attn_layer, "kv_cache"):
        raise RuntimeError("No kv_cache on attention layer")
    kv_cache = attn_layer.kv_cache[ve]
    key_cache, value_cache = kv_cache[0], kv_cache[1]  # paged blocks
    if not isinstance(key_cache, torch.Tensor):
        key_cache = torch.as_tensor(key_cache)
    if not isinstance(value_cache, torch.Tensor):
        value_cache = torch.as_tensor(value_cache)
    # Merge blocks → sequence
    keys_seq   = key_cache.reshape(-1, key_cache.shape[-2], key_cache.shape[-1])
    values_seq = value_cache.reshape(-1, value_cache.shape[-2], value_cache.shape[-1])
    return keys_seq, values_seq  # [T, H_kv, Dh]

def populated_len(keys_seq):
    """
    Count populated tokens (non-zero rows) in [T, H_kv, Dh].
    Handles partially-filled last block.
    """
    with torch.no_grad():
        mask = (keys_seq.abs().sum(dim=(1,2)) > 0)
        return int(mask.sum().item())

def flatten_kv(seq):  # [T, H_kv, Dh] -> [T, D]
    return seq.reshape(seq.shape[0], -1)

def mean_token_cosine(A_2d, B_2d):
    """
    Mean cosine over tokens between two [T, D] matrices (row-wise).
    """
    A = F.normalize(A_2d, dim=1)
    B = F.normalize(B_2d, dim=1)
    return (A * B).sum(dim=1).mean().item()

def build_kv_similarity(K_flat_list, V_flat_list, wK=0.5, wV=0.5, contiguity_penalty=0.0):
    """
    Build per-layer similarity matrices S_K, S_V, and blended S.
    - K_flat_list / V_flat_list: list of [T_common, D] tensors (one per layer)
    - contiguity_penalty: subtract a small amount proportional to |i-j| to bias contiguity (0 = off)
    """
    L = len(K_flat_list)
    S_K = torch.zeros((L, L), dtype=torch.float32)
    S_V = torch.zeros((L, L), dtype=torch.float32)
    for i in range(L):
        S_K[i, i] = 1.0
        S_V[i, i] = 1.0
        for j in range(i+1, L):
            sk = mean_token_cosine(K_flat_list[i], K_flat_list[j])
            sv = mean_token_cosine(V_flat_list[i], V_flat_list[j])
            if contiguity_penalty > 0.0:
                gap = abs(i - j)
                penalty = min(0.5, contiguity_penalty * gap)
                sk = max(0.0, sk - penalty)
                sv = max(0.0, sv - penalty)
            S_K[i, j] = S_K[j, i] = sk
            S_V[i, j] = S_V[j, i] = sv
    S = (wK * S_K + wV * S_V).clamp(0.0, 1.0)
    return S_K, S_V, S

def clusters_by_threshold(S, sim_threshold=0.90):
    """
    Graph clustering via thresholding S (non-contiguous allowed).
    Returns connected components as clusters (lists of layer indices 0..L-1).
    """
    L = S.shape[0]
    adj = [[] for _ in range(L)]
    for i in range(L):
        for j in range(i+1, L):
            if S[i, j].item() >= sim_threshold:
                adj[i].append(j)
                adj[j].append(i)
    visited = [False]*L
    clusters = []
    for i in range(L):
        if visited[i]:
            continue
        q = [i]; visited[i] = True; comp = [i]
        while q:
            u = q.pop()
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    q.append(v); comp.append(v)
        clusters.append(sorted(comp))
    return clusters

def choose_anchor_for_cluster(cluster, S):
    """
    Pick the layer with highest average in-cluster similarity (diag excluded).
    """
    if len(cluster) == 1:
        return cluster[0]
    sub = S[cluster][:, cluster].clone()
    idx = torch.arange(sub.shape[0])
    sub[idx, idx] = 0.0
    means = sub.mean(dim=1)
    return cluster[int(torch.argmax(means).item())]

def top_pairs(sim_mat, layers, topk=5):
    """
    Return top-K off-diagonal similarity pairs from sim_mat.
    Each pair is (layer0idx, layer0idx, score). layers is [0..L-1].
    """
    L = sim_mat.shape[0]
    if L < 2:
        return []
    triu_idx = torch.triu_indices(L, L, offset=1)
    if triu_idx.numel() == 0:
        return []
    scores = sim_mat[triu_idx[0], triu_idx[1]]
    k = min(int(topk), int(scores.numel()))
    if k == 0:
        return []
    vals, order = torch.topk(scores, k=k)
    pairs = []
    for j in range(k):
        orig_idx = int(order[j].item())
        i1 = int(triu_idx[0, orig_idx].item())
        i2 = int(triu_idx[1, orig_idx].item())
        score = float(vals[j].item())
        pairs.append((layers[i1], layers[i2], score))
    pairs.sort(key=lambda x: x[2], reverse=True)
    return pairs

# ---------- extract KV for ALL layers and align tokens ----------
VE = 0                    # virtual engine index
MAX_TOKENS = 256          # cap for efficiency; use larger if your prompt is long
wK, wV = 0.5, 0.5         # blend weights
SIM_THRESHOLD = 0.015    # cluster threshold (raise for stricter clusters)
CONTIGUITY_PENALTY = 0.01 # small bias toward contiguous groups if desired (e.g., 0.01–0.03)

# Gather sequences
K_seq = {}
V_seq = {}
pop_lens = []
usable_layers = []
for lid in layer_ids:
    try:
        k, v = get_kv_seq(attn_layers[lid], ve=VE)  # [T, H_kv, Dh]
    except Exception as e:
        print(f"  Skipping layer {lid+1}: {e}")
        continue
    # limit tokens
    T = min(k.shape[0], MAX_TOKENS)
    k = k[:T].contiguous()
    v = v[:T].contiguous()
    # populated length
    T_pop = min(populated_len(k), populated_len(v))
    if T_pop == 0:
        print(f"  Skipping layer {lid+1}: empty KV")
        continue
    K_seq[lid] = k
    V_seq[lid] = v
    pop_lens.append(T_pop)
    usable_layers.append(lid)

if len(usable_layers) < 2:
    raise RuntimeError("Not enough layers with populated KV to compute similarity.")

common_T = min(pop_lens)
print(f"\nAligned token length across layers: {common_T} tokens (cap {MAX_TOKENS}).")
usable_layers = sorted(usable_layers)

# Flatten per layer to [T_common, D]
K_flat_list = []
V_flat_list = []
for lid in usable_layers:
    K_flat_list.append(flatten_kv(K_seq[lid][:common_T]).float().cpu())
    V_flat_list.append(flatten_kv(V_seq[lid][:common_T]).float().cpu())

# ---------- build similarity matrices ----------
S_K, S_V, S = build_kv_similarity(
    K_flat_list, V_flat_list,
    wK=wK, wV=wV, contiguity_penalty=CONTIGUITY_PENALTY
)

# ---------- report top pairs ----------
pairs_k = top_pairs(S_K, list(range(len(usable_layers))), topk=8)
pairs_v = top_pairs(S_V, list(range(len(usable_layers))), topk=8)
pairs_b = top_pairs(S,   list(range(len(usable_layers))), topk=8)

def fmt_pairs(pairs):
    lines = []
    for a, b, s in pairs:
        # Map back to real layer numbers (1-indexed for readability)
        la = usable_layers[a] + 1
        lb = usable_layers[b] + 1
        lines.append(f"Layers {la} ↔ {lb}: cos sim = {s:.4f}")
    return "\n".join(lines) if lines else "(none)"

print("\n=== TOP K-similarity layer pairs ===")
print(fmt_pairs(pairs_k))
print("\n=== TOP V-similarity layer pairs ===")
print(fmt_pairs(pairs_v))
print("\n=== TOP Blended(K,V) layer pairs ===")
print(fmt_pairs(pairs_b))

# ---------- cluster by threshold & choose anchors ----------
clusters_idx = clusters_by_threshold(S, sim_threshold=SIM_THRESHOLD)
anchors_idx = [choose_anchor_for_cluster(c, S) for c in clusters_idx]

# Build sharing plan: anchor provides KV to others in its cluster
sharing_plan = {}
for c, a in zip(clusters_idx, anchors_idx):
    targets = [t for t in c if t != a]
    if targets:
        sharing_plan[a] = targets

def one_index(lst): return [usable_layers[i] + 1 for i in lst]

print("\n=== Clusters (1-indexed layer numbers) ===")
for c in clusters_idx:
    print("  ", one_index(c))

print("\nAnchors per cluster (1-indexed):", one_index(anchors_idx))

print("\n=== Sharing plan (1-indexed) ===")
if not sharing_plan:
    print("(no pairs exceed threshold; lower SIM_THRESHOLD to form clusters)")
else:
    for a, ts in sharing_plan.items():
        print(f"  Layer {usable_layers[a]+1} → {one_index(ts)}")

# ---------- cleanup request (optional) ----------
try:
    engine.abort_request(req_id)
except Exception:
    pass

print("\n✅ Per-layer KV similarity clustering complete.")
